# Huấn luyện YOLO11n phát hiện phương tiện

Đưa `vehicle-colab-pilot-v1.zip` vào thư mục `detect-vehicle` trong My Drive. Chọn GPU T4 trong **Runtime → Change runtime type**, sau đó chọn **Runtime → Run all**.

Notebook huấn luyện từ đầu, chọn `best.pt` theo tập validation và đánh giá trên tập test sau khi train. Checkpoint và báo cáo được lưu trong `My Drive/detect-vehicle/runs/`.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU trong Runtime → Change runtime type, rồi chạy lại.')
print(torch.cuda.get_device_name(0))
drive.mount('/content/drive')
drive_project = Path('/content/drive/MyDrive/detect-vehicle')
archive = drive_project / 'vehicle-colab-pilot-v1.zip'
if not archive.is_file():
    raise FileNotFoundError(f'Chưa tìm thấy gói dữ liệu tại {archive}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.216'], check=True)

In [ ]:
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
from zipfile import ZipFile

epochs = 80
image_size = 640
batch = 8
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run_dir = drive_project / 'runs' / f'pilot-{run_id}'

with TemporaryDirectory(dir='/content') as temp:
    work = Path(temp)
    with ZipFile(archive) as bundle:
        bundle.extractall(work)
    subprocess.run([
        sys.executable, '-u', str(work / 'train_colab.py'),
        '--dataset', str(work / 'dataset'), '--output', str(run_dir),
        '--epochs', str(epochs), '--imgsz', str(image_size), '--batch', str(batch),
    ], check=True)

In [ ]:
print((run_dir / 'test_metrics.json').read_text(encoding='utf-8'))
print(f'Checkpoint: {run_dir / "best.pt"}')